# Maxwell–Boltzmann speed distribution

**Prerequisite:** [Maxwell–Boltzmann speed distribution](../../../docs/lecture-notes/statistical-mechanics/maxwell-boltzmann-speed/index.md)

This maintained computational laboratory consolidates earlier gas and statistical-mechanics explorations. Reusable state validation, characteristic speeds, and speed-density evaluation now belong to `projectkoios.physkit`.

## Learning objectives

1. Configure a Maxwell–Boltzmann gas state with explicit SI magnitudes.
2. Evaluate and normalize its speed probability density.
3. Compare the most probable, mean, and root-mean-square speeds.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.statmech.kinetic_theory import (
    MaxwellBoltzmannGasState,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

In [ ]:
temperature_kelvin = 300.0
nitrogen_molar_mass_kg_per_mol = 0.0280134
plot_speed_maximum_metres_per_second = 2000.0
normalization_speed_maximum_metres_per_second = 5000.0
plot_sample_count = 20001
normalization_sample_count = 50001
speed_unit = PhysicalUnit(expression="meter / second")

nitrogen_300_kelvin = MaxwellBoltzmannGasState(
    temperature=ScalarQuantity(
        magnitude=temperature_kelvin,
        unit=PhysicalUnit(expression="kelvin"),
    ),
    molar_mass=ScalarQuantity(
        magnitude=nitrogen_molar_mass_kg_per_mol,
        unit=PhysicalUnit(expression="kilogram / mole"),
    ),
)
speed_values = np.linspace(
    0.0,
    plot_speed_maximum_metres_per_second,
    plot_sample_count,
    dtype=np.float64,
)
speeds = VectorQuantity(magnitude=speed_values, unit=speed_unit)
density = nitrogen_300_kelvin.evaluate_speed_distribution(
    speeds=speeds
).probability_density
normalization_speed_values = np.linspace(
    0.0,
    normalization_speed_maximum_metres_per_second,
    normalization_sample_count,
    dtype=np.float64,
)
normalization_speeds = VectorQuantity(
    magnitude=normalization_speed_values,
    unit=speed_unit,
)
normalization_density = nitrogen_300_kelvin.evaluate_speed_distribution(
    speeds=normalization_speeds
).probability_density
represented_probability = np.trapezoid(
    normalization_density.magnitude,
    x=normalization_speed_values,
)

characteristic_speeds = {
    "most probable": nitrogen_300_kelvin.most_probable_speed,
    "mean": nitrogen_300_kelvin.mean_speed,
    "root mean square": nitrogen_300_kelvin.root_mean_square_speed,
}
represented_probability, characteristic_speeds

In [ ]:
normalization_tolerance = 2.0e-14
normalization_holds = np.isclose(
    represented_probability,
    1.0,
    rtol=0.0,
    atol=normalization_tolerance,
)
characteristic_speed_order_holds = (
    characteristic_speeds["most probable"].magnitude
    < characteristic_speeds["mean"].magnitude
    < characteristic_speeds["root mean square"].magnitude
)

assert normalization_holds
assert characteristic_speed_order_holds
normalization_holds, characteristic_speed_order_holds

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(speed_values, density.magnitude, label="speed density")
for name, speed in characteristic_speeds.items():
    axis.axvline(speed.magnitude, linestyle="--", label=name)
axis.set(
    xlabel="speed $v$ (m/s)",
    ylabel="probability density $f(v)$ (s/m)",
    title="Nitrogen Maxwell–Boltzmann speed distribution at 300 K",
)
axis.set_xlim(0.0, plot_speed_maximum_metres_per_second)
axis.set_ylim(bottom=0.0)
axis.grid(True)
axis.legend()
plt.show()

## Interpretation

The calculated density is normalized on the resolved finite speed interval to the displayed numerical tolerance. The vertical lines show the expected ordering $v_{\mathrm{mp}} < \langle vangle < v_{\mathrm{rms}}$ for the represented model.

This calculation assumes a dilute classical gas in thermal equilibrium. Agreement with the analytic model is numerical verification of the implementation, not validation that nitrogen under arbitrary conditions obeys those assumptions.

## Exercises

1. Repeat the calculation for another positive molar mass and temperature.
2. Increase the upper speed bound and study the finite-domain normalization error.
3. Explain dimensionally why the density has units of seconds per metre.
